# EDA: выгрузка «Альфа-Курс» (Трек 1, рекомендательная система)

**Цель:** понять, какие таблицы и поля есть в выгрузке, сколько в них строк и пропусков, и какие данные пригодны для рекомендаций.

Данные приходят как PostgreSQL-дамп (`.backup`). PostgreSQL ставить не нужно: `src/backup_reader.py` читает дамп напрямую.
Положите файл в `data/raw/alfa_course_hackathon.backup` (папка в `.gitignore`, исходные данные в GitHub не загружаем).

In [ ]:
import sys, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 220)

sys.path.append(str(Path("../src").resolve()))   # запускать ноутбук из папки notebooks/
from backup_reader import BackupReader

BACKUP = Path("../data/raw/alfa_course_hackathon.backup")
assert BACKUP.exists(), f"Положите дамп в {BACKUP.resolve()}"
br = BackupReader(BACKUP)
T = br.load_all()          # {имя_таблицы: DataFrame}, все значения пока строки
print(f"таблиц: {len(T)}, строк всего: {sum(len(d) for d in T.values()):,}")

## 1. Что есть: таблицы, строки, столбцы, пропуски

In [ ]:
GROUPS = {
    "каталог": ["Articles","Courses","Lessons","Documents","Webinars","Podcasts","Glossary","Calculators"],
    "содержимое": ["ArticleEmbedItems","LessonEmbedItems","DocumentEmbedItems","GlossaryEmbedItems"],
    "активность": ["CourseUserBindings","CourseUserLessonBindings","UserRatings","UserRatingTagsBindings"],
    "метаданные/связи": ["ContentSettings","ContentSettingsTopics","ContentSettingsTags","ContentSettingsIndustries",
                         "Topics","Tags","Industries","ArticleSubjects","RecommendedItems","CourseLessonBindings"],
    "тарифы/подписки": ["CourseSubscriptions","CourseTariffBindings","Subscription","SubscriptionTariffs"],
}
t2g = {t: g for g, ts in GROUPS.items() for t in ts}

inv = pd.DataFrame({
    t: {"группа": t2g.get(t, "прочее"), "строк": len(d), "столбцов": d.shape[1],
        "полностью пустых столбцов": int((d.isna().all()).sum()),
        "пропусков в ячейках, %": round(d.isna().mean().mean() * 100, 1)}
    for t, d in T.items()
}).T.sort_values(["группа", "строк"], ascending=[True, False])
inv

## 2. Столбцы ключевых таблиц

Показываем тип данных, долю пропусков, число уникальных значений и пример. Технические поля (`CreatedById`, `IsDeleted`, `Order` и т.п.) можно сразу отбросить, они почти всегда константны или пусты.

In [ ]:
def describe_cols(df):
    return pd.DataFrame({
        "missing_%": (df.isna().mean() * 100).round(0),
        "nunique": df.nunique(),
        "example": [str(df[c].dropna().iloc[0])[:45] if df[c].notna().any() else None for c in df.columns],
    })

for t in ["Articles", "Courses", "Lessons", "Documents", "Webinars", "Podcasts"]:
    print("=" * 90, f"\n{t}: {len(T[t]):,} строк")
    display(describe_cols(T[t]))

## 3. Справочники и единый каталог материалов

In [ ]:
access = T["CourseTypes"].set_index("Id")["Title"].to_dict()     # Открытый / Платный / По подписке / Закрытый
status = T["Statuses"].set_index("Id")["Title"].to_dict()        # Опубликовано / Черновик / ...

KINDS = {"article": "Articles", "course": "Courses", "document": "Documents", "webinar": "Webinars", "podcast": "Podcasts"}
parts = []
for kind, tname in KINDS.items():
    d = T[tname]
    x = d[["Id", "Title"] + [c for c in ["Duration"] if c in d.columns]].copy()
    x["kind"] = kind
    parts.append(x)
catalog = pd.concat(parts, ignore_index=True)

cs = T["ContentSettings"].rename(columns={"Id": "cs_id"})[["cs_id", "EntityId", "TypeId", "StatusId", "Published"]]
catalog = catalog.merge(cs, left_on="Id", right_on="EntityId", how="left")
catalog["access"] = catalog.TypeId.map(access)
catalog["status"] = catalog.StatusId.map(status)

# темы / теги / отрасли в виде списков названий
def attach(link, key, ref, ref_title="Title", name="x"):
    m = T[link].merge(T[ref][["Id", ref_title]], left_on=key, right_on="Id")
    return m.groupby("ContentSettingsId")[ref_title].apply(list).rename(name)

for link, key, ref, name in [("ContentSettingsTopics", "TopicId", "Topics", "topics"),
                             ("ContentSettingsTags", "TagId", "Tags", "tags"),
                             ("ContentSettingsIndustries", "IndustryId", "Industries", "industries")]:
    catalog = catalog.merge(attach(link, key, ref, name=name), left_on="cs_id", right_index=True, how="left")
    catalog[name] = catalog[name].apply(lambda v: v if isinstance(v, list) else [])
    catalog[f"n_{name}"] = catalog[name].str.len()

def to_minutes(s):
    if not isinstance(s, str): return np.nan
    h = re.search(r"(\d+)\s*час", s); m = re.search(r"(\d+)\s*мин", s)
    return (int(h.group(1)) * 60 if h else 0) + (int(m.group(1)) if m else 0) or np.nan
catalog["duration_min"] = catalog.Duration.map(to_minutes)

print("материалов в каталоге:", len(catalog))
catalog.pivot_table(index="kind", columns=["access", "status"], values="Id", aggfunc="count", fill_value=0)

### Платные и бесплатные: что реально можно рекомендовать к покупке

In [ ]:
c = T["Courses"].copy()
c["access_by_Courses"] = c.TypeId.map(access)
cmp = c.merge(catalog[catalog.kind == "course"][["Id", "access"]], on="Id", how="left")
print("Курсы: тип доступа по Courses.TypeId против ContentSettings.TypeId")
display(pd.crosstab(cmp.access_by_Courses, cmp.access))

print("\nКурсы с ценой (Courses.Cost):")
display(c[c.Cost.notna()][["Title", "Cost", "access_by_Courses"]])
print("Тарифы подписки:")
display(T["SubscriptionTariffs"][["Title", "Cost", "DiscountCost", "Recurrence", "RecurrenceValue"]])
print("Курсы в подписке (CourseSubscriptions):", len(T["CourseSubscriptions"]))

## 4. Метаданные материалов: чем можно описать материал

In [ ]:
pub = catalog[catalog.status == "Опубликовано"]
cov = pub.groupby("kind").agg(
    n=("Id", "size"),
    with_topics=("n_topics", lambda s: (s > 0).mean()),
    with_tags=("n_tags", lambda s: (s > 0).mean()),
    with_industries=("n_industries", lambda s: (s > 0).mean()),
    with_duration=("duration_min", lambda s: s.notna().mean()),
).round(2)
display(cov)
ax = cov[["with_topics", "with_tags", "with_industries", "with_duration"]].plot.bar(figsize=(9, 3.5), title="Доля опубликованных материалов с заполненным полем")
ax.set_ylim(0, 1.05); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

print("Справочники: тем", len(T["Topics"]), "| тегов", len(T["Tags"]), "| отраслей", len(T["Industries"]))
print("Реально используются: тем", T["ContentSettingsTopics"].TopicId.nunique(),
      "| тегов", T["ContentSettingsTags"].TagId.nunique(),
      "| отраслей", T["ContentSettingsIndustries"].IndustryId.nunique())

topic_cnt = pub.explode("topics").dropna(subset=["topics"]).groupby("topics").size().sort_values(ascending=False)
topic_cnt.head(15)

## 5. Тексты материалов (для content-based и поиска)

In [ ]:
def text_len(embed, key):
    s = T[embed][[key, "HtmlContent"]].copy()
    s["chars"] = s.HtmlContent.fillna("").str.replace(r"<[^>]+>", " ", regex=True).str.len()
    return s.groupby(key).chars.sum()

art_len = text_len("ArticleEmbedItems", "ArticleId")
doc_len = text_len("DocumentEmbedItems", "DocumentId")
les_len = text_len("LessonEmbedItems", "LessonId")
print(f"статей с текстом: {len(art_len)} из {len(T['Articles'])}; медиана ~{int(art_len.median())} символов")
print(f"документов с текстом: {len(doc_len)} из {len(T['Documents'])}")
print(f"уроков с текстом: {len(les_len)} из {len(T['Lessons'])} (включая полные тексты платных уроков, их нельзя раскрывать пользователю)")
art_len.clip(upper=60000).hist(bins=40, figsize=(7, 3)); plt.title("символов в статье (обрезано на 60k)"); plt.show()

## 6. Пользовательская активность

В выгрузке нет отдельной таблицы пользователей: есть только обезличенный `UserId` в записях о курсах и оценках. Анкеты (стадия бизнеса, отрасль, опыт) нет.

In [ ]:
cub = T["CourseUserBindings"].copy()
for col in ["Created", "Completed", "StudyingStartDate", "Modified"]:
    cub[col] = pd.to_datetime(cub[col], utc=True, errors="coerce")
cub["paid"] = cub.IsPaid.eq("t"); cub["done"] = cub.IsCompleted.eq("t")

n_user = cub.UserId.nunique()
per_user = cub.groupby("UserId").size()
print(f"записей: {len(cub):,} | пользователей: {n_user:,} | курсов с записями: {cub.CourseId.nunique()} из {len(T['Courses'])}")
print(f"пользователей с одной записью: {(per_user == 1).mean():.1%} | с 2+ курсами: {(per_user >= 2).sum():,}")
print(f"оплачено записей: {cub.paid.sum()} ({cub.paid.mean():.2%}), пользователей с оплатой: {cub[cub.paid].UserId.nunique()}")
print(f"завершено курсов: {cub.done.sum():,} ({cub.done.mean():.1%})")

fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))
per_user.clip(upper=10).value_counts().sort_index().plot.bar(ax=axs[0], title="курсов на пользователя (10 = 10+)")
pop = cub.groupby("CourseId").size().sort_values(ascending=False)
(pop.cumsum() / pop.sum()).reset_index(drop=True).plot(ax=axs[1], title="накопленная доля записей по курсам (популярность)")
axs[1].set_xlabel("курсы по убыванию"); plt.tight_layout(); plt.show()
print(f"топ-1 курс: {pop.iloc[0] / len(cub):.1%} записей, топ-5: {pop.head(5).sum() / len(cub):.1%}")

cub[["ContentAccess", "Status"]].apply(lambda s: s.value_counts()).fillna(0).astype(int)

### Насколько точные даты

In [ ]:
clb = T["CourseUserLessonBindings"].copy()
clb["Completed"] = pd.to_datetime(clb.Completed, utc=True)

for name, s in [("CourseUserBindings.Created", cub.Created), ("CourseUserBindings.Modified", cub.Modified),
                ("CourseUserLessonBindings.Completed", clb.Completed)]:
    s = s.dropna()
    print(f"{name}: {s.min().date()} -> {s.max().date()} | уникальных дат: {s.dt.date.nunique()} | "
          f"всегда 1-е число: {(s.dt.day == 1).all()} | время суток: {s.dt.strftime('%H:%M').unique()[:3]}")
cub.groupby(cub.Created.dt.to_period("M")).size().plot.bar(figsize=(9, 3), title="новые записи на курс по месяцам")
plt.tight_layout(); plt.show()

### Прохождение уроков и оценки

In [ ]:
cu_user = cub.set_index("Id").UserId
clb["UserId"] = clb.CourseUserId.map(cu_user)
print(f"уроков пройдено (записей): {len(clb):,} | пользователей: {clb.UserId.nunique():,} | уроков с прогрессом: {clb.LessonId.nunique()} из {len(T['Lessons'])}")
print("пользователей с 1-2 пройденными уроками:", f"{(clb.groupby('UserId').size() <= 2).mean():.1%}")

ur = T["UserRatings"].copy(); ur["Value"] = ur.Value.astype(int)
ent = {}
for k, n in [("Lessons", "lesson"), ("Calculators", "calculator")]:
    ent.update({i: n for i in T[k].Id})
ur["entity"] = ur.EntityId.map(ent)
print(f"\nоценок: {len(ur):,} | с UserId: {ur.UserId.notna().mean():.1%} | уникальных пользователей: {ur.UserId.nunique():,}")
display(ur.groupby("entity").agg(n=("Id", "size"), mean_value=("Value", "mean"), with_user=("UserId", lambda s: s.notna().mean())).round(2))
print("распределение оценок:", ur.Value.value_counts().sort_index().to_dict())
print("оценщики, которые есть и в CourseUserBindings:", len(set(ur.UserId.dropna()) & set(cub.UserId)), "из", ur.UserId.nunique())

### RecommendedItems: готовые связки «материал -> похожие»

In [ ]:
ri = T["RecommendedItems"].copy()
kind_by_id = catalog.set_index("Id").kind.to_dict()
ri["kind"] = ri.ItemId.map(kind_by_id)
print(f"строк: {len(ri):,} | групп (Id): {ri.Id.nunique():,} | материалов: {ri.ItemId.nunique()}")
print("элементов в группе:", ri.groupby("Id").size().describe().round(1).to_dict())
display(ri.kind.fillna("другое (калькулятор и т.п.)").value_counts())
ri.head()

## 7. Холодный старт в цифрах

In [ ]:
hist = per_user.rename("n_courses").to_frame()
hist["has_lessons"] = hist.index.isin(clb.UserId.dropna().unique())
print("пользователей всего:", len(hist))
print("с историей из 2+ курсов (можно делать leave-one-out):", int((hist.n_courses >= 2).sum()))
print("с хотя бы одним пройденным уроком:", int(hist.has_lessons.sum()))
print(f"материалов без единого действия пользователя: статьи/документы/вебинары/подкасты = "
      f"{int(catalog.kind.isin(['article','document','webinar','podcast']).sum())} (событий по ним в выгрузке нет)")

## 8. Выводы

**Объём и структура.** 62 таблицы, 102 128 строк. Каталог: 855 статей, 37 курсов, 508 уроков, 247 документов, 19 вебинаров, 8 подкастов, 351 термин глоссария, 7 калькуляторов. Содержимое: 32 475 блоков текста статей, 2 892 блока уроков, 274 блока документов, 5 756 блоков глоссария.

**Главное для проекта**

1. **Покупки почти не представлены.** Платный курс всего один (3 990 ₽, тариф «Стандарт»), ещё 4–5 курсов (источники расходятся) и 5 подкастов «по подписке» (подписка 2 990 ₽/мес и акционные тарифы 990 и 495 ₽). В `CourseUserBindings` только 22 оплаченные записи из 10 050 (0,2%). Обучить модель на покупке нельзя. Предлагаем прокси-цели: старт курса, прохождение уроков, завершение, переход к подписке, и ранжирование с учётом правил (платное предлагаем только после релевантного бесплатного).
2. **События есть только по курсам и урокам.** По статьям, документам, вебинарам и подкастам действий пользователей нет вообще. Для 1 129 материалов вне курсов (статьи, документы, вебинары, подкасты) работает только content-based (темы, теги, отрасли, текст) и связки `RecommendedItems`.
3. **Истории короткие.** 7 342 пользователя, у 79% одна запись о курсе, у 21% (около 1,5 тыс.) два и больше. У большинства история короткая, поэтому нужен холодный старт: онбординг и стартовая подборка.
4. **Профиля пользователя нет.** Ни стадии бизнеса, ни отрасли, ни опыта. Эти поля придётся собирать в онбординге и сопоставлять с темами, тегами, отраслями и типом материала.
5. **Даты грубые.** Все даты событий приведены к 1-му числу месяца, 12:00 (13 месяцев для записей на курсы, 29 дат для прохождения уроков). Метрику «дни возврата» по этим данным посчитать нельзя; в плане проверки эффекта нужно предусмотреть собственное логирование дат захода. Временное разбиение возможно только по месяцам. Даты прохождения уроков начинаются с 2024-05, раньше первых записей на курсы (2025-09), так что часть истории, вероятно, перенесена из старой системы.
6. **Оценки.** 8 343 оценок, 84% из них пятёрки, 72% без `UserId` (оценки калькуляторов анонимны полностью, оценки уроков привязаны к пользователю примерно в половине случаев); оцениваются только уроки и калькуляторы, статьи, документы и курсы не оцениваются. Как сигнал качества слабый, как явный сигнал о пользователе почти бесполезен.
7. **Популярность сильно сконцентрирована.** Топ-5 курсов дают около 60% всех записей; популярное будет сильным бейзлайном, поэтому качество стоит мерить и на «хвосте» каталога.

**Какие данные полезны для рекомендаций**

| Что | Где | Зачем |
|---|---|---|
| Единый каталог (тип, название, статус, доступ) | `Articles`, `Courses`, `Documents`, `Webinars`, `Podcasts` + `ContentSettings` (`TypeId`, `StatusId`) | кандидаты, фильтр «опубликовано», бесплатно/по подписке/платно |
| Темы, теги, отрасли | `ContentSettingsTopics/Tags/Industries` + справочники | content-based, сопоставление с онбордингом, объяснения |
| Время чтения/просмотра | `Duration` (статьи, курсы, уроки, вебинары, подкасты) | подбор под формат «быстрый ответ» или «подробный курс» |
| Тип урока | `Lessons.TypeId` (видео, тест, лонгрид) | формат обучения |
| Цена и подписка | `Courses.Cost`, `CourseTariffBindings`, `SubscriptionTariffs`, `CourseSubscriptions` | предложение платного, объяснение пользы |
| Структура курса и порядок уроков | `CourseLessonBindings` | маршрут обучения, «следующий шаг» |
| Тексты | `*EmbedItems.HtmlContent`, `Description`, заголовки | поиск по запросу, эмбеддинги, TF-IDF |
| История пользователя | `CourseUserBindings`, `CourseUserLessonBindings` | collaborative и последовательные сигналы, leave-one-out оценка |
| Явные оценки | `UserRatings` (только уроки и калькуляторы) | слабый сигнал качества материала |
| Готовые связки «похожее» | `RecommendedItems` | дополнительный бейзлайн и проверка item-item |

**Что не использовать или использовать осторожно**
- Тексты платных уроков (`LessonEmbedItems`): по условиям кейса нельзя раскрывать содержимое платных материалов, брать только краткую пользу и ссылки.
- Расхождение доступа между `Courses.TypeId` и `ContentSettings.TypeId` у одного курса (открытый против «по подписке», см. раздел 3): выбрать один источник и уточнить у организаторов.
- Технические столбцы (`CreatedById`, `ModifiedById`, `IsDeleted`, `Order`, `ExternalCss/Html/Js` и др.): константы или пусты.

**Следующий шаг:** собрать единый каталог (`catalog` из раздела 3), сделать бейзлайн по совпадению темы и популярному, затем гибрид content-based плюс правила холодного старта; оценивать на leave-one-out по пользователям с 2+ курсами и на тестовых профилях.